# LeetCode 3549 — Multiply Two Polynomials bằng Python

- Link: https://leetcode.com/problems/multiply-two-polynomials/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: **Tích chập** (convolution), **chia để trị Karatsuba**, **FFT**, **thế Kronecker** (đóng gói đa thức vào số nguyên lớn)

> **Ghi chú Premium:** đề bài và chữ ký hàm được viết lại theo bản mô tả công khai trên doocs (leetcode.doocs.org) vì nội dung gốc chỉ dành cho tài khoản Premium. Chữ ký `multiply(self, poly1, poly2)` suy ra từ code Python trên doocs — nên đối chiếu lại nếu có tài khoản Premium.

Skeleton (Python3, theo doocs):

```python
class Solution:
    def multiply(self, poly1: List[int], poly2: List[int]) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu nhân hai đa thức = **tích chập** hai dãy hệ số: `r[k] = Σ_{i+j=k} a[i]·b[j]`.
2. Cài cách trực tiếp `O(n·m)` và thấy vì sao nó quá chậm khi `n = m = 5·10^4`.
3. Hiểu ý tưởng **Karatsuba**: 3 phép nhân nửa kích thước thay cho 4 ⇒ `O(n^1.585)`.
4. Hiểu ý tưởng **FFT**: đánh giá đa thức tại các căn đơn vị, nhân từng điểm, nội suy ngược ⇒ `O(n log n)`; lưu ý sai số số thực.
5. Biết mẹo rất "Python": **thế Kronecker** — đóng gói hệ số vào một số nguyên lớn, để phép nhân số lớn (cài bằng C) làm tích chập **chính xác**, xử lý cả hệ số âm.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `poly1`, `poly2`; phần tử thứ `i` là hệ số của `x^i`. Gọi `A(x)`, `B(x)` là hai đa thức tương ứng. Trả về mảng `result` dài `len(poly1) + len(poly2) − 1` là hệ số của `R(x) = A(x)·B(x)` (giữ cả hệ số 0 ở cuối).

**Ví dụ 1:** `poly1 = [3,2,5]`, `poly2 = [1,4]` → `[3,14,13,20]` (`(3 + 2x + 5x²)(1 + 4x) = 3 + 14x + 13x² + 20x³`).

**Ví dụ 2:** `poly1 = [1,0,-2]`, `poly2 = [-1]` → `[-1,0,2]`.

**Ví dụ 3:** `poly1 = [1,5,-3]`, `poly2 = [-4,2,0]` → `[-4,-18,22,-6,0]`.

**Ràng buộc (theo doocs):** `1 ≤ len(poly1), len(poly2) ≤ 5·10^4`, `−10^3 ≤ hệ số ≤ 10^3`, mỗi đa thức có ít nhất một hệ số khác 0.

In [1]:
from typing import List
import math
examples = [
    ([3, 2, 5], [1, 4], [3, 14, 13, 20]),
    ([1, 0, -2], [-1], [-1, 0, 2]),
    ([1, 5, -3], [-4, 2, 0], [-4, -18, 22, -6, 0]),
]
for a, b, exp in examples:
    print(a, '*', b, '-> kỳ vọng', exp)

[3, 2, 5] * [1, 4] -> kỳ vọng [3, 14, 13, 20]
[1, 0, -2] * [-1] -> kỳ vọng [-1, 0, 2]
[1, 5, -3] * [-4, 2, 0] -> kỳ vọng [-4, -18, 22, -6, 0]


## 3. Kiến thức nền

### 3.1. Đa thức là mảng hệ số

`[3, 2, 5]` ↔ `3 + 2x + 5x²`. Đánh giá tại `x` bằng sơ đồ Horner: `((5)·x + 2)·x + 3`.

In [2]:
def evaluate(p, x):
    r = 0
    for c in reversed(p):
        r = r * x + c
    return r

print(evaluate([3, 2, 5], 2), 3 + 2 * 2 + 5 * 4)

27 27


### 3.2. Tích chập

Hệ số `x^k` trong tích = tổng `a[i]·b[j]` với `i + j = k`. Với `len(a) = n`, `len(b) = m` thì có `n + m − 1` hệ số.

In [3]:
a, b = [3, 2, 5], [1, 4]
for k in range(len(a) + len(b) - 1):
    terms = [(i, k - i) for i in range(len(a)) if 0 <= k - i < len(b)]
    print(f"x^{k}: " + ' + '.join(f"a[{i}]·b[{j}]" for i, j in terms), '=', sum(a[i] * b[j] for i, j in terms))

x^0: a[0]·b[0] = 3
x^1: a[0]·b[1] + a[1]·b[0] = 14
x^2: a[1]·b[1] + a[2]·b[0] = 13
x^3: a[2]·b[1] = 20


### 3.3. Số phức, căn đơn vị (cho FFT)

`ω = e^{2πi/N} = cos(2π/N) + i·sin(2π/N)`; các lũy thừa `ω^0, ω^1, …, ω^{N−1}` là `N` điểm cách đều trên đường tròn đơn vị. Python có sẵn kiểu `complex` và `cmath`.

In [4]:
import cmath
N = 4
w = cmath.exp(2j * math.pi / N)
print([complex(round((w ** k).real, 6), round((w ** k).imag, 6)) for k in range(N)])

[(1+0j), 1j, (-1+0j), (-0-1j)]


### 3.4. Số nguyên lớn và `int.to_bytes` / `int.from_bytes`

Python có số nguyên không giới hạn, phép nhân số lớn được cài bằng C (thuật toán Karatsuba). `int.from_bytes(b, 'little')` ghép các byte thành số; `n.to_bytes(k, 'little')` tách ngược lại — cả hai chạy tuyến tính, rất nhanh.

In [5]:
x = int.from_bytes(bytes([1, 2, 3]), 'little')
print(x, x == 1 + 2 * 256 + 3 * 256 ** 2, list(x.to_bytes(3, 'little')))

197121 True [1, 2, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 3: `(1 + 5x − 3x²)(−4 + 2x + 0x²)`.

| k | các cặp | giá trị |
|---|---|---|
| 0 | 1·(−4) | −4 |
| 1 | 1·2 + 5·(−4) | −18 |
| 2 | 1·0 + 5·2 + (−3)·(−4) | 22 |
| 3 | 5·0 + (−3)·2 | −6 |
| 4 | (−3)·0 | 0 |

Quan sát: đây đúng là phép **nhân số theo cột** như ở tiểu học, chỉ là **không nhớ** sang cột sau. Nếu "cột" đủ rộng để không bao giờ cần nhớ, phép nhân số nguyên lớn sẽ cho ra đúng các hệ số — đó là ý tưởng thế Kronecker ở cách 5.

In [6]:
# Nhân "theo cột" với cơ số 10^6: các hệ số nhỏ nên không bị nhớ chồng
a, b = [3, 2, 5], [1, 4]
BASE = 10**6
A = evaluate(a, BASE)
Bv = evaluate(b, BASE)
P = A * Bv
digits = []
while P:
    digits.append(P % BASE)
    P //= BASE
print('A =', A, '| B =', Bv, '| các "chữ số" của A·B theo cơ số 10^6:', digits)

A = 5000002000003 | B = 4000001 | các "chữ số" của A·B theo cơ số 10^6: [3, 14, 13, 20]


## 5. Cách 1 — Brute Force: hai vòng lặp

Cộng dồn `a[i]·b[j]` vào `r[i + j]`.

In [7]:
def multiply_naive(a, b):
    r = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        if x == 0:
            continue
        for j, y in enumerate(b):
            r[i + j] += x * y
    return r

for a, b, exp in examples:
    print(multiply_naive(a, b), '| kỳ vọng', exp)

[3, 14, 13, 20] | kỳ vọng [3, 14, 13, 20]
[-1, 0, 2] | kỳ vọng [-1, 0, 2]
[-4, -18, 22, -6, 0] | kỳ vọng [-4, -18, 22, -6, 0]


**Độ phức tạp:** `O(n·m)` = `2.5·10^9` phép nhân với `n = m = 5·10^4` — quá chậm.

**Nút thắt:** mỗi cặp `(i, j)` được nhân riêng. Cần cách "nhân theo khối" (Karatsuba), "đổi miền" (FFT) hoặc "giao cho phép nhân số lớn tối ưu sẵn" (Kronecker).

## 6. Ý tưởng thuật toán / bước đột phá

**Karatsuba:** tách `A = A0 + A1·x^h`, `B = B0 + B1·x^h`. Khi đó

`A·B = A0B0 + ((A0 + A1)(B0 + B1) − A0B0 − A1B1)·x^h + A1B1·x^{2h}`

— chỉ **3** phép nhân nửa kích thước ⇒ `T(n) = 3T(n/2) + O(n) = O(n^{log2 3}) ≈ O(n^1.585)`.

**FFT:** một đa thức bậc `< N` xác định duy nhất bởi giá trị tại `N` điểm. Chọn `N` điểm là căn đơn vị bậc `N` (lũy thừa của 2): đánh giá tại mọi điểm chỉ tốn `O(N log N)` nhờ tách chẵn/lẻ `A(x) = E(x²) + x·O(x²)`. Nhân từng điểm, rồi FFT ngược (chia `N`) để lấy lại hệ số. Kết quả số thực ⇒ làm tròn.

**Thế Kronecker:** chọn `X = 2^k` đủ lớn để mỗi hệ số của tích nằm gọn trong `k` bit (có thêm "độ lệch" để xử lý số âm). Khi đó `R(X) = A(X)·B(X)` là **một** phép nhân số nguyên lớn, và các khối `k` bit của `R(X)` chính là hệ số. Với ràng buộc `|r_k| ≤ 5·10^4 · 10^6 = 5·10^{10} < 2^{39}`, chọn `k = 40` bit (5 byte).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Karatsuba đệ quy

In [8]:
def add_poly(p, q):
    if len(p) < len(q):
        p, q = q, p
    r = p[:]
    for i, v in enumerate(q):
        r[i] += v
    return r

def karatsuba(a, b):
    if min(len(a), len(b)) <= 32:
        return multiply_naive(a, b)
    if len(a) < len(b):
        a, b = b, a                            # a là đa thức dài hơn
    h = len(a) // 2
    if len(b) <= h:                            # b quá ngắn: chỉ tách a
        lo, hi = karatsuba(a[:h], b), karatsuba(a[h:], b)
        r = [0] * (len(a) + len(b) - 1)
        for i, v in enumerate(lo):
            r[i] += v
        for i, v in enumerate(hi):
            r[i + h] += v
        return r
    a0, a1 = a[:h], a[h:]
    b0, b1 = b[:h], b[h:]
    z0 = karatsuba(a0, b0)
    z2 = karatsuba(a1, b1)
    z1 = karatsuba(add_poly(a0, a1), add_poly(b0, b1))
    for i, v in enumerate(z0):
        z1[i] -= v
    for i, v in enumerate(z2):
        z1[i] -= v
    r = [0] * (len(a) + len(b) - 1)
    for i, v in enumerate(z0):
        r[i] += v
    for i, v in enumerate(z1):
        if i + h < len(r):
            r[i + h] += v
    for i, v in enumerate(z2):
        r[i + 2 * h] += v
    return r

for a, b, exp in examples:
    print(karatsuba(a, b), '| kỳ vọng', exp)

[3, 14, 13, 20] | kỳ vọng [3, 14, 13, 20]
[-1, 0, 2] | kỳ vọng [-1, 0, 2]
[-4, -18, 22, -6, 0] | kỳ vọng [-4, -18, 22, -6, 0]


Chú ý `z1` có thể dài hơn khoảng cần thiết với các số 0 ở cuối (do `A0 + A1` dài `max(h, n − h)`); các phần tử vượt độ dài kết quả đều bằng 0 nên bỏ qua.

**Độ phức tạp:** `O(n^1.585)`; với `n = 5·10^4` vẫn khoảng `3·10^7` phép nhân trong Python thuần — vài chục giây.

### 7.2. Cách 3 — FFT số phức (theo lời giải tham khảo trên doocs)

In [9]:
def fft(a, invert):
    n = len(a)
    j = 0
    for i in range(1, n):                      # hoán vị đảo bit
        bit = n >> 1
        while j & bit:
            j ^= bit
            bit >>= 1
        j ^= bit
        if i < j:
            a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:
        ang = 2 * math.pi / length * (-1 if invert else 1)
        wlen = complex(math.cos(ang), math.sin(ang))
        half = length >> 1
        for i in range(0, n, length):
            w = 1 + 0j
            for k in range(i, i + half):
                u = a[k]
                v = a[k + half] * w
                a[k] = u + v
                a[k + half] = u - v
                w *= wlen
        length <<= 1
    if invert:
        for i in range(n):
            a[i] /= n

def multiply_fft(p1, p2):
    m = len(p1) + len(p2) - 1
    n = 1
    while n < m:
        n <<= 1
    fa = [complex(x) for x in p1] + [0j] * (n - len(p1))
    fb = [complex(x) for x in p2] + [0j] * (n - len(p2))
    fft(fa, False)
    fft(fb, False)
    for i in range(n):
        fa[i] *= fb[i]
    fft(fa, True)
    return [int(round(fa[i].real)) for i in range(m)]

for a, b, exp in examples:
    print(multiply_fft(a, b), '| kỳ vọng', exp)

[3, 14, 13, 20] | kỳ vọng [3, 14, 13, 20]
[-1, 0, 2] | kỳ vọng [-1, 0, 2]
[-4, -18, 22, -6, 0] | kỳ vọng [-4, -18, 22, -6, 0]


**Độ phức tạp:** `O(N log N)` với `N` là lũy thừa 2 ≥ `n + m − 1`. Trong Python thuần mỗi bước "bướm" (butterfly) là vài phép tính số phức, nên `N = 2^17` mất khoảng một giây (xem 12.3). Sai số: giá trị kết quả ≤ `5·10^{10}`, sai số dấu phẩy động cỡ `10^{-4}` ≪ 0.5 ⇒ làm tròn an toàn.

### 7.3. Cách 4 — Thế Kronecker với số nguyên lớn (lời giải chính)

1. Đóng gói mỗi hệ số vào 5 byte (40 bit). Hệ số âm: tách thành phần dương và phần âm, `A = A⁺ − A⁻`.
2. `P = A·B` (một phép nhân số lớn).
3. Cộng độ lệch `OFF = 2^39` vào **mọi** "chữ số" (một số nguyên lặp mẫu 5 byte) ⇒ mọi chữ số thành không âm và `< 2^40`, nên không có nhớ/mượn giữa các khối.
4. `to_bytes`, cắt từng khối 5 byte, trừ `OFF`.

In [10]:
K = 5                     # số byte mỗi hệ số
OFF = 1 << 39             # độ lệch: |r_k| <= 5e10 < 2^39

def pack(p):
    pos = b''.join((x if x > 0 else 0).to_bytes(K, 'little') for x in p)
    neg = b''.join((-x if x < 0 else 0).to_bytes(K, 'little') for x in p)
    return int.from_bytes(pos, 'little') - int.from_bytes(neg, 'little')

def multiply_kronecker(p1, p2):
    m = len(p1) + len(p2) - 1
    P = pack(p1) * pack(p2)
    P += int.from_bytes(OFF.to_bytes(K, 'little') * m, 'little')
    raw = P.to_bytes(K * m + 1, 'little')
    return [int.from_bytes(raw[K * i:K * i + K], 'little') - OFF for i in range(m)]

for a, b, exp in examples:
    print(multiply_kronecker(a, b), '| kỳ vọng', exp)

[3, 14, 13, 20] | kỳ vọng [3, 14, 13, 20]
[-1, 0, 2] | kỳ vọng [-1, 0, 2]
[-4, -18, 22, -6, 0] | kỳ vọng [-4, -18, 22, -6, 0]


**Vì sao 40 bit đủ:** mỗi hệ số tích có `|r_k| ≤ min(n, m)·10^3·10^3 ≤ 5·10^{10}`. Sau khi cộng `2^39 ≈ 5.5·10^{11}`, giá trị nằm trong `(0, 2^40)`. Vì mọi "chữ số" `r_k + OFF` thuộc `[0, 2^40)`, biểu diễn cơ số `2^40` của `P + OFF·Σ 2^{40k}` **chính là** dãy `r_k + OFF` (không có nhớ).

**Độ phức tạp:** phép nhân số lớn `~2·10^6` bit dùng Karatsuba trong C: vài phần mười giây.

## 8. TRACE từng bước

### 8.1. Trace Karatsuba ở mức đầu tiên (đa thức nhỏ, ngưỡng thấp)

In [11]:
def trace_karatsuba(a, b, depth=0, limit=1):
    pad = '   ' * depth
    if min(len(a), len(b)) <= limit:
        r = multiply_naive(a, b)
        print(f"{pad}nhân trực tiếp {a} * {b} = {r}")
        return r
    h = max(len(a), len(b)) // 2
    a0, a1, b0, b1 = a[:h], a[h:], b[:h], b[h:]
    print(f"{pad}tách h={h}: A0={a0}, A1={a1}, B0={b0}, B1={b1}")
    z0 = trace_karatsuba(a0, b0, depth + 1, limit)
    z2 = trace_karatsuba(a1, b1, depth + 1, limit)
    z1 = trace_karatsuba(add_poly(a0, a1), add_poly(b0, b1), depth + 1, limit)
    mid = [v - (z0[i] if i < len(z0) else 0) - (z2[i] if i < len(z2) else 0) for i, v in enumerate(z1)]
    r = [0] * (len(a) + len(b) - 1)
    for i, v in enumerate(z0): r[i] += v
    for i, v in enumerate(mid):
        if i + h < len(r): r[i + h] += v
    for i, v in enumerate(z2): r[i + 2 * h] += v
    print(f"{pad}z0={z0}, z2={z2}, z1-z0-z2={mid} -> {r}")
    return r

trace_karatsuba([1, 2, 3, 4], [5, 6, 7, 8])
print('đối chiếu:', multiply_naive([1, 2, 3, 4], [5, 6, 7, 8]))

tách h=2: A0=[1, 2], A1=[3, 4], B0=[5, 6], B1=[7, 8]
   tách h=1: A0=[1], A1=[2], B0=[5], B1=[6]
      nhân trực tiếp [1] * [5] = [5]
      nhân trực tiếp [2] * [6] = [12]
      nhân trực tiếp [3] * [11] = [33]
   z0=[5], z2=[12], z1-z0-z2=[16] -> [5, 16, 12]
   tách h=1: A0=[3], A1=[4], B0=[7], B1=[8]
      nhân trực tiếp [3] * [7] = [21]
      nhân trực tiếp [4] * [8] = [32]
      nhân trực tiếp [7] * [15] = [105]
   z0=[21], z2=[32], z1-z0-z2=[52] -> [21, 52, 32]
   tách h=1: A0=[4], A1=[6], B0=[12], B1=[14]
      nhân trực tiếp [4] * [12] = [48]
      nhân trực tiếp [6] * [14] = [84]
      nhân trực tiếp [10] * [26] = [260]
   z0=[48], z2=[84], z1-z0-z2=[128] -> [48, 128, 84]
z0=[5, 16, 12], z2=[21, 52, 32], z1-z0-z2=[22, 60, 40] -> [5, 16, 34, 60, 61, 52, 32]
đối chiếu: [5, 16, 34, 60, 61, 52, 32]


### 8.2. Trace thế Kronecker (ví dụ 3): in từng khối 5 byte

In [12]:
def trace_kronecker(p1, p2):
    m = len(p1) + len(p2) - 1
    A, Bv = pack(p1), pack(p2)
    print('A(2^40) =', A)
    print('B(2^40) =', Bv)
    P = A * Bv
    print('A·B     =', P)
    P += int.from_bytes(OFF.to_bytes(K, 'little') * m, 'little')
    raw = P.to_bytes(K * m + 1, 'little')
    out = []
    for i in range(m):
        chunk = raw[K * i:K * i + K]
        v = int.from_bytes(chunk, 'little')
        out.append(v - OFF)
        print(f"  khối {i}: bytes={chunk.hex()} -> {v} - 2^39 = {v - OFF}")
    return out

trace_kronecker([1, 5, -3], [-4, 2, 0])

A(2^40) = -3626777458838389965979647
B(2^40) = 2199023255548
A·B     = -7975367974682898869391340311047831556
  khối 0: bytes=fcffffff7f -> 549755813884 - 2^39 = -4
  khối 1: bytes=eeffffff7f -> 549755813870 - 2^39 = -18
  khối 2: bytes=1600000080 -> 549755813910 - 2^39 = 22
  khối 3: bytes=faffffff7f -> 549755813882 - 2^39 = -6
  khối 4: bytes=0000000080 -> 549755813888 - 2^39 = 0


[-4, -18, 22, -6, 0]

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def multiply(self, poly1: List[int], poly2: List[int]) -> List[int]:
        K = 5
        OFF = 1 << 39
        m = len(poly1) + len(poly2) - 1

        def pack(p):
            pos = b''.join((x if x > 0 else 0).to_bytes(K, 'little') for x in p)
            neg = b''.join((-x if x < 0 else 0).to_bytes(K, 'little') for x in p)
            return int.from_bytes(pos, 'little') - int.from_bytes(neg, 'little')

        P = pack(poly1) * pack(poly2)
        P += int.from_bytes(OFF.to_bytes(K, 'little') * m, 'little')
        raw = P.to_bytes(K * m + 1, 'little')
        return [int.from_bytes(raw[K * i:K * i + K], 'little') - OFF for i in range(m)]

for a, b, exp in examples:
    got = Solution().multiply(a, b)
    print(got, 'OK' if got == exp else 'SAI')

[3, 14, 13, 20] OK
[-1, 0, 2] OK
[-4, -18, 22, -6, 0] OK


## 10. Vì sao thuật toán đúng

1. **Tích chập = nhân đa thức** theo định nghĩa phân phối.
2. **Karatsuba:** `(A0 + A1)(B0 + B1) = A0B0 + A0B1 + A1B0 + A1B1` ⇒ phần giữa `A0B1 + A1B0` thu được bằng phép trừ, đúng với mọi đa thức.
3. **FFT:** đánh giá tại `N ≥ n + m − 1` điểm phân biệt xác định duy nhất đa thức tích (bậc `< N`); FFT ngược là nội suy chính xác (về mặt toán học). Sai số số thực nhỏ hơn 0.5 với ràng buộc đề ⇒ làm tròn cho đúng số nguyên.
4. **Kronecker:** với `X = 2^40`, `A(X)·B(X) = R(X) = Σ r_k X^k`. Cộng `OFF·Σ X^k` cho `Σ (r_k + OFF) X^k` với `0 ≤ r_k + OFF < X` ⇒ đây chính là biểu diễn cơ số `X` duy nhất của số đó ⇒ đọc khối `k` cho `r_k + OFF`. Đóng gói `A⁺ − A⁻` đúng bằng `A(X)` kể cả hệ số âm.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(poly1)`, `m = len(poly2)`, `N` = lũy thừa 2 ≥ `n + m − 1`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hai vòng lặp | `O(nm)` | `O(n + m)` | chính xác |
| 2. Karatsuba | `O(N^{1.585})` | `O(N)` | chính xác, Python thuần chậm |
| 3. FFT | `O(N log N)` | `O(N)` | số thực, cần làm tròn |
| 4. Kronecker | `O(M(40N))` với `M` là chi phí nhân số lớn (Karatsuba trong C) | `O(N)` byte | chính xác, nhanh nhất trong Python |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {
    'naive': multiply_naive,
    'karatsuba': karatsuba,
    'fft': multiply_fft,
    'kronecker': multiply_kronecker,
    'submit': lambda a, b: Solution().multiply(a, b),
}
edge = [
    ([5], [7], [35]),
    ([0, 0, 1], [1], [0, 0, 1]),                  # x^2 * 1
    ([-1000], [-1000], [10**6]),
    ([1000] * 3, [-1000] * 2, [-10**6, -2 * 10**6, -2 * 10**6, -10**6]),
    ([1, -1], [1, 1], [1, 0, -1]),                # (1 - x)(1 + x)
]
for a, b, exp in examples + edge:
    for name, f in impls.items():
        got = f(a, b)
        assert got == exp, (name, a, b, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên (kích thước tới 200)

In [15]:
import random
random.seed(3549)
cnt = 0
for _ in range(300):
    a = [random.randint(-1000, 1000) for _ in range(random.randint(1, 200))]
    b = [random.randint(-1000, 1000) for _ in range(random.randint(1, 200))]
    exp = multiply_naive(a, b)
    for name, f in impls.items():
        assert f(a, b) == exp, (name, len(a), len(b))
    cnt += 1
print('Khớp cách trực tiếp trên', cnt, 'cặp đa thức ngẫu nhiên.')

Khớp cách trực tiếp trên 300 cặp đa thức ngẫu nhiên.


### 12.3. Ca lớn (`5·10^4 × 5·10^4`) và ca giá trị cực trị

In [16]:
import time
random.seed(1)
n = 5 * 10**4
cases = {
    'ngẫu nhiên': ([random.randint(-1000, 1000) for _ in range(n)], [random.randint(-1000, 1000) for _ in range(n)]),
    'toàn 1000 × toàn -1000': ([1000] * n, [-1000] * n),
}
for name, (a, b) in cases.items():
    t = time.time(); r1 = Solution().multiply(a, b); t1 = time.time() - t
    t = time.time(); r2 = multiply_fft(a, b); t2 = time.time() - t
    assert r1 == r2
    # kiểm tra độc lập: so vài hệ số với tổng trực tiếp
    for k in [0, 1, n - 1, n, 2 * n - 2]:
        assert r1[k] == sum(a[i] * b[k - i] for i in range(max(0, k - n + 1), min(k, n - 1) + 1))
    print(f"{name}: Kronecker {t1:.2f}s | FFT Python {t2:.2f}s | min/max hệ số {min(r1)} / {max(r1)}")

ngẫu nhiên: Kronecker 0.33s | FFT Python 0.88s | min/max hệ số -338729046 / 297314374


toàn 1000 × toàn -1000: Kronecker 0.33s | FFT Python 0.82s | min/max hệ số -50000000000 / -1000000


Ca "toàn 1000 × toàn −1000" đẩy hệ số giữa tới `−5·10^{10}` — biên của độ lệch `2^39` — và vẫn chính xác.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — FFT không làm tròn (ép `int` cắt cụt phần thập phân)

In [17]:
def multiply_fft_truncate(p1, p2):
    m = len(p1) + len(p2) - 1
    n = 1
    while n < m:
        n <<= 1
    fa = [complex(x) for x in p1] + [0j] * (n - len(p1))
    fb = [complex(x) for x in p2] + [0j] * (n - len(p2))
    fft(fa, False); fft(fb, False)
    for i in range(n):
        fa[i] *= fb[i]
    fft(fa, True)
    return [int(fa[i].real) for i in range(m)]        # SAI: 13.9999999 -> 13

random.seed(2)
a = [random.randint(-1000, 1000) for _ in range(300)]
b = [random.randint(-1000, 1000) for _ in range(300)]
bad = sum(x != y for x, y in zip(multiply_fft_truncate(a, b), multiply_naive(a, b)))
print('số hệ số sai khi cắt cụt:', bad, '/', len(a) + len(b) - 1)

số hệ số sai khi cắt cụt: 525 / 599


**Vì sao sai:** kết quả FFT là số thực xấp xỉ như `13.999999999` hoặc `−6.0000000001`; `int()` cắt về 0 nên lệch 1. Phải `round`.

### Lỗi 2 — Thế Kronecker với khối quá hẹp (bị nhớ sang khối bên cạnh)

In [18]:
def kron_narrow(p1, p2, nbytes):
    off = 1 << (8 * nbytes - 1)
    m = len(p1) + len(p2) - 1
    def pk(p):
        pos = b''.join((x if x > 0 else 0).to_bytes(nbytes, 'little') for x in p)
        neg = b''.join((-x if x < 0 else 0).to_bytes(nbytes, 'little') for x in p)
        return int.from_bytes(pos, 'little') - int.from_bytes(neg, 'little')
    P = pk(p1) * pk(p2) + int.from_bytes(off.to_bytes(nbytes, 'little') * m, 'little')
    raw = P.to_bytes(nbytes * m + 8, 'little')
    return [int.from_bytes(raw[nbytes * i:nbytes * i + nbytes], 'little') - off for i in range(m)]

a, b = [1000] * 100, [1000] * 100          # hệ số giữa = 10^8 > 2^23
print('3 byte  :', kron_narrow(a, b, 3)[99], '| đúng:', multiply_naive(a, b)[99])
print('5 byte  :', kron_narrow(a, b, 5)[99])

3 byte  : -663290 | đúng: 100000000
5 byte  : 100000000


**Vì sao sai:** mỗi khối phải chứa được **toàn bộ** miền giá trị `[−5·10^{10}, 5·10^{10}]` sau khi cộng độ lệch; khối 3 byte chỉ chứa `2^24 ≈ 1.6·10^7`.

### Lỗi 3 — Đóng gói hệ số âm trực tiếp bằng `to_bytes` (không có `signed=True`)

In [19]:
try:
    (-3).to_bytes(5, 'little')
except OverflowError as e:
    print('OverflowError:', e)
print('cách đúng: tách A = A⁺ - A⁻ ->', pack([1, 5, -3]) == 1 + 5 * 2**40 - 3 * 2**80)

OverflowError: can't convert negative int to unsigned
cách đúng: tách A = A⁺ - A⁻ -> True


**Vì sao sai:** `to_bytes` mặc định không nhận số âm. Có thể dùng `signed=True` nhưng khi ghép bằng `from_bytes` lại sai nghĩa (bù 2 trên từng khối). Tách phần dương/âm là cách gọn và đúng.

### Lỗi 4 — Bỏ các hệ số 0 ở cuối kết quả

In [20]:
def strip_trailing(r):
    while len(r) > 1 and r[-1] == 0:
        r = r[:-1]                     # SAI: đề yêu cầu đúng độ dài len1 + len2 - 1
    return r

r = Solution().multiply([1, 5, -3], [-4, 2, 0])
print('sai :', strip_trailing(r))
print('đúng:', r)

sai : [-4, -18, 22, -6]
đúng: [-4, -18, 22, -6, 0]


**Vì sao sai:** ví dụ 3 của đề giữ `0` ở cuối — kết quả phải có đúng `len(poly1) + len(poly2) − 1` phần tử.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng `a[i]·b[j]` với `i + j = k`" ⇒ **tích chập** ⇒ FFT/NTT/Karatsuba.
- Trong Python: hệ số nguyên, giới hạn biết trước ⇒ **thế Kronecker** cho kết quả chính xác và rất nhanh.
- FFT số phức: luôn `round`, kiểm tra biên độ lớn nhất × `N` × `log N` × `ε` ≪ 0.5.
- Cần modulo nguyên tố dạng `c·2^k + 1` (như `998244353`) ⇒ **NTT** thay cho FFT số phức.
- Bài tập đếm tổng hai phần tử (LC 1010-kiểu "đếm cặp có tổng s" trên miền nhỏ) ⇒ tích chập histogram.

## 15. Biến thể của bài

1. Nhân hai **số nguyên rất lớn** cho dưới dạng chuỗi (LC 43 *Multiply Strings*) ⇒ tích chập chữ số rồi xử lý nhớ.
2. Đếm số cặp `(i, j)` có `a[i] + b[j] = s` cho mọi `s` ⇒ tích chập hai histogram.
3. Nhân đa thức modulo `998244353` ⇒ NTT.
4. Lũy thừa đa thức `A(x)^k` ⇒ bình phương liên tiếp với tích chập.

In [21]:
# Biến thể 2: đếm cặp tổng bằng tích chập histogram (dùng Kronecker)
def pair_sum_counts(xs, ys, maxv):
    hx = [0] * (maxv + 1)
    hy = [0] * (maxv + 1)
    for x in xs:
        hx[x] += 1
    for y in ys:
        hy[y] += 1
    return multiply_kronecker(hx, hy)

xs, ys = [1, 2, 2, 5], [0, 3, 3]
cnt = pair_sum_counts(xs, ys, 5)
brute = [sum(1 for x in xs for y in ys if x + y == s) for s in range(11)]
assert cnt == brute
print('số cặp theo tổng:', cnt)

số cặp theo tổng: [0, 1, 2, 0, 2, 5, 0, 0, 2, 0, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Chính xác? | Khi nào dùng |
|---|---|---|---|---|
| 1 | Hai vòng lặp | `O(nm)` | có | `n·m ≤ 10^7` |
| 2 | Karatsuba | `O(N^{1.585})` | có | Ngôn ngữ biên dịch, không có FFT |
| 3 | FFT số phức | `O(N log N)` | xấp xỉ, làm tròn | Chuẩn mực trong C++/Java |
| 4 | Thế Kronecker | ≈ nhân số lớn | có | **Nộp bài bằng Python** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `multiply([1, 1], [1, 1])` = ?

**Bài 2 (tính tay):** `multiply([2, 0, 1], [0, 3])` = ?

**Bài 3 (điền TODO):** viết `evaluate_product_check(p1, p2, r, x)` kiểm tra `A(x)·B(x) == R(x)` tại một điểm.

**Bài 4 (tự viết lại):** LC 43 — nhân hai chuỗi số không âm bằng tích chập chữ số + xử lý nhớ.

**Bài 5 (mini challenge):** viết **NTT** modulo `998244353` và dùng hai modulo + CRT (hoặc kiểm tra biên) để nhân đa thức hệ số nhỏ.

In [22]:
def evaluate_product_check_todo(p1, p2, r, x):
    # TODO
    raise NotImplementedError

def multiply_strings_todo(s, t):
    # TODO
    raise NotImplementedError

def ntt_multiply_todo(p1, p2):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `(1 + x)^2`. (2) `1 + 2x + x²`. (3) `[1, 2, 1]`.

**Bài 2.** (1) `(2 + x²)(3x)`. (2) `6x + 3x³`. (3) `[0, 6, 0, 3]`.

**Bài 3.** (1) Horner cho ba đa thức. (2) So sánh. (3) Thử nhiều `x` ngẫu nhiên (kiểm tra kiểu Schwartz–Zippel).

**Bài 4.** (1) Đảo chuỗi thành mảng chữ số. (2) Tích chập (dùng Kronecker). (3) Xử lý nhớ từ hàng đơn vị, bỏ số 0 đầu.

**Bài 5.** (1) Căn nguyên thủy của `998244353` là 3. (2) Cấu trúc giống FFT, thay `complex` bằng phép nhân modulo. (3) Hệ số âm: đưa về modulo, sau đó đổi lại về `(−p/2, p/2]` (đủ khi |r| < p/2).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
a1 = Solution().multiply([1, 1], [1, 1])
a2 = Solution().multiply([2, 0, 1], [0, 3])
assert a1 == [1, 2, 1] and a2 == [0, 6, 0, 3]

def evaluate_product_check(p1, p2, r, x):
    return evaluate(p1, x) * evaluate(p2, x) == evaluate(r, x)

random.seed(3)
for _ in range(50):
    a = [random.randint(-9, 9) for _ in range(random.randint(1, 20))]
    b = [random.randint(-9, 9) for _ in range(random.randint(1, 20))]
    r = Solution().multiply(a, b)
    assert all(evaluate_product_check(a, b, r, x) for x in range(-3, 4))
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: kiểm tra tại điểm đúng')

Bài 1: [1, 2, 1] | Bài 2: [0, 6, 0, 3] | Bài 3: kiểm tra tại điểm đúng


In [24]:
# Bài 4: LC 43 Multiply Strings
def multiply_strings(s, t):
    if s == "0" or t == "0":
        return "0"
    a = [int(c) for c in reversed(s)]
    b = [int(c) for c in reversed(t)]
    r = Solution().multiply(a, b)
    carry, out = 0, []
    for v in r:
        carry += v
        out.append(carry % 10)
        carry //= 10
    while carry:
        out.append(carry % 10)
        carry //= 10
    while len(out) > 1 and out[-1] == 0:
        out.pop()
    return ''.join(map(str, reversed(out)))

random.seed(4)
for _ in range(200):
    x, y = random.randint(0, 10**30), random.randint(0, 10**30)
    assert multiply_strings(str(x), str(y)) == str(x * y)
print('Bài 4:', multiply_strings("123", "456"))

Bài 4: 56088


In [25]:
# Bài 5: NTT modulo 998244353
MODN, G = 998244353, 3

def ntt(a, invert):
    n = len(a)
    j = 0
    for i in range(1, n):
        bit = n >> 1
        while j & bit:
            j ^= bit
            bit >>= 1
        j ^= bit
        if i < j:
            a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:
        w = pow(G, (MODN - 1) // length, MODN)
        if invert:
            w = pow(w, MODN - 2, MODN)
        half = length >> 1
        for i in range(0, n, length):
            wn = 1
            for k in range(i, i + half):
                u, v = a[k], a[k + half] * wn % MODN
                a[k] = (u + v) % MODN
                a[k + half] = (u - v) % MODN
                wn = wn * w % MODN
        length <<= 1
    if invert:
        inv = pow(n, MODN - 2, MODN)
        for i in range(n):
            a[i] = a[i] * inv % MODN

def ntt_multiply(p1, p2):
    m = len(p1) + len(p2) - 1
    n = 1
    while n < m:
        n <<= 1
    fa = [x % MODN for x in p1] + [0] * (n - len(p1))
    fb = [x % MODN for x in p2] + [0] * (n - len(p2))
    ntt(fa, False); ntt(fb, False)
    fa = [x * y % MODN for x, y in zip(fa, fb)]
    ntt(fa, True)
    return [v - MODN if v > MODN // 2 else v for v in fa[:m]]   # đúng khi |r| < MODN/2

random.seed(5)
for _ in range(100):
    a = [random.randint(-1000, 1000) for _ in range(random.randint(1, 100))]
    b = [random.randint(-1000, 1000) for _ in range(random.randint(1, 100))]
    assert ntt_multiply(a, b) == multiply_naive(a, b)      # |r| <= 100 * 10^6 < MODN/2
print('Bài 5: NTT khớp trên hệ số đủ nhỏ; ví dụ', ntt_multiply([1, 5, -3], [-4, 2, 0]))

Bài 5: NTT khớp trên hệ số đủ nhỏ; ví dụ [-4, -18, 22, -6, 0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Thế Kronecker (Python): hệ số tích |r| < 2^39
pack(p) = from_bytes(phần dương, 5 byte/hệ số) - from_bytes(phần âm)
P = pack(a) * pack(b) + from_bytes(OFF.to_bytes(5) * m)      # OFF = 2^39
raw = P.to_bytes(5*m + 1); r[i] = from_bytes(raw[5i:5i+5]) - OFF
# FFT: pad lên lũy thừa 2 >= n+m-1, fft, nhân từng điểm, fft ngược, ROUND
```

**Checklist:**

- [ ] Độ dài kết quả đúng `len1 + len2 − 1` (giữ số 0 cuối)?
- [ ] Khối đủ rộng cho `|r| ≤ 5·10^{10}` (40 bit) và có độ lệch cho số âm?
- [ ] FFT: dùng `round`, không cắt `int`?
- [ ] Hệ số âm khi đóng gói số lớn được tách `A⁺ − A⁻`?
- [ ] Đã đối chiếu với cách hai vòng lặp trên dữ liệu nhỏ?